# SwiftRoute Logistics - Delivery Delay Prediction
## DSAI Mini Coursework

**Student name**: Thanh Diem Nguyen 

**Email**: thanhdiem0512@gmail.com 

**Discord username**: sophiedn  

**Submission date**: *(fill in)*

---

### Mark Scheme

| Section | Marks |
|---|---|
| 1 | 10 |
| 2 | 3 |
| 3 | 10 |
| 4 | 12 |
| 5 | 15 |
| 6 | 15 |
| 7 | 8 |
| 8 | 12 |
| 9 | 5 |
| Extension | 10 |
| **Total** | **100** |

---
# SECTION 1 - Understanding the Problem *(10 marks)*

**1.1** *(1 mark)* In one sentence, describe what SwiftRoute is trying to predict and why it matters.

*Your answer here.*

---

**1.2** *(2 marks)* Is this a classification or regression problem? What is the target variable? What does one row in the training data represent?

*Your answer here.*

---

**1.3** *(3 marks)* Only ~12% of shipments are delayed. Explain why **accuracy** is a misleading metric here. What metric will you use instead, and what does it measure in the context of delivery operations?

*Your answer here.*

---

**1.4** *(2 marks)* The Operations team can only intervene on the top **15%** of highest-risk shipments. Describe the two types of mistake your model can make, and explain which is more costly to SwiftRoute.

*Your answer here.*

---

**1.5** *(2 marks)* The `weather_events` table stores disruptions as date ranges (start_date to end_date) by region. Explain the challenge of joining this to `shipments`, and what logic is needed.

*Your answer here.*

---
# SECTION 2 - Setup & Data Loading *(3 marks)*

In [3]:
# *(1 mark)* Import libraries
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42

In [7]:
# *(2 marks)* Load all four CSV files. Print shape and first few rows of each.
# shipments = pd.read_csv('data/shipments.csv', parse_dates=['pickup_date'])
# routes    = pd.read_csv('data/routes.csv')
# carriers  = pd.read_csv('data/carriers.csv')
# weather   = pd.read_csv('data/weather_events.csv', parse_dates=['start_date','end_date'])

# Your code here
from swiftroute.dataset import load_dataset, check_dataset

data = load_dataset()

for name, df in data.items():
    print(f'{name}: {df.shape}')
    display(df.head(5))

shipments: (10025, 12)


,shipment_id,pickup_date,origin_region,dest_region,service_tier,carrier_id,route_id,weight_kg,volume_cm3,declared_value,is_business,is_delayed
0,SH0000001,2025-09-26,London,North_West,economy,C03,R0038,11.54,9427,5.44,0,0
1,SH0000002,2024-12-09,North_West,North_East,standard,C01,R0044,9.26,9864,7.48,1,0
2,SH0000003,2025-11-15,South_East,South_West,express,C05,R0013,0.43,96,37.44,0,0
3,SH0000004,2025-06-12,South_East,North_East,express,C08,R0009,8.84,6541,141.25,0,1
4,SH0000005,2026-03-21,Yorkshire,Scotland,standard,C04,R0010,11.10,4959,148.71,0,0


routes: (60, 7)


,route_id,origin_region,dest_region,distance_km,n_depot_hops,historical_delay_rate,avg_transit_days
0,R0001,London,North_East,480,3,0.2471,2.92
1,R0002,Scotland,Yorkshire,139,1,0.1617,1.85
2,R0003,South_West,Midlands,191,1,0.1449,1.71
3,R0004,London,South_East,127,1,0.0450,1.85
4,R0005,Midlands,North_East,279,2,0.0970,1.96


carriers: (8, 7)


,carrier_id,carrier_name,tier_coverage,reliability_score,capacity_utilisation,avg_delay_mins,n_active_routes
0,C01,FastTrack,all,0.9508,0.5926,78.2,17
1,C02,PrimeMove,all,0.9049,0.6838,82.2,25
2,C03,SureDeliver,all,0.8605,0.6675,35.0,8
3,C04,QuickHaul,all,0.8332,0.7361,29.7,11
4,C05,NorthernPost,express_only,0.7640,0.8150,34.9,15


weather: (265, 6)


,event_id,region,event_type,severity,start_date,end_date
0,W00001,Midlands,high_wind,2,2024-01-03,2024-01-05
1,W00002,Yorkshire,snow,1,2024-01-26,2024-01-30
2,W00003,London,flood,1,2024-01-15,2024-01-18
3,W00004,North_East,high_wind,1,2024-01-26,2024-02-01
4,W00005,North_West,storm,2,2024-01-24,2024-01-26


---
# SECTION 3 - Exploring the Data (EDA) *(10 marks)*

### 3.1 Data Quality Check *(3 marks)*

In [ ]:
# *(3 marks)* Check: dtypes, null counts, duplicates, referential integrity,
# and the overall delay rate.

# Your code here
check_dataset(data)

**Commentary** - List quality issues and how you will handle each.

*Your answer here.*

### 3.2 Distribution of Key Columns *(2 marks)*

In [ ]:
# *(2 marks)* Plot: service_tier mix, weight distribution, delay rate over time
# (monthly), carrier reliability scores, route distance distribution.

# Your code here

**Commentary** - Two interesting observations.

*Your answer here.*

### 3.3 Which Factors Are Related to Delays? *(3 marks)*

In [ ]:
# *(3 marks)* Explore delay rate vs:
# - service_tier
# - carrier (join to carriers.csv for reliability_score)
# - route distance (join to routes.csv)
# - weather (do shipments during weather events delay more?)

# Your code here

**Commentary** - Which factors show the strongest relationship with delays?

*Your answer here.*

### 3.4 Which Routes and Carriers Are Riskiest? *(2 marks)*

In [ ]:
# *(2 marks)* Show: top 10 highest-delay routes, delay rate by carrier,
# and a heatmap of delay rate by origin x destination region.

# Your code here

**Commentary** - Which two route/carrier combinations are riskiest? Business implication?

*Your answer here.*

---
# SECTION 4 - Cleaning & Joining the Data *(12 marks)*

In [ ]:
# *(2 marks)* 4.1 - Remove duplicate shipments. Keep first occurrence.

# Your code here

In [ ]:
# *(4 marks)* 4.2 - Join routes and carriers to shipments.
# Bring in: distance_km, n_depot_hops, historical_delay_rate, avg_transit_days
# from routes; reliability_score, capacity_utilisation from carriers.

# Your code here

In [ ]:
# *(6 marks)* 4.3 - Join weather events.
# For each shipment, determine the maximum weather severity affecting its
# origin region and destination region on its pickup_date.
# Hint: a shipment at row i is weather-affected at origin if there exists any
# weather event where region == origin_region AND start_date <= pickup_date <= end_date.

# Your code here
# master = ...

---
# SECTION 5 - Creating New Features *(15 marks)*

In [ ]:
# *(9 marks)* Create at least 5 new features. Comment each with its rationale.
#
# Suggested directions:
# - weather_affected: binary flag - any weather at origin OR destination
# - weather_both_ends: both origin AND destination affected
# - is_winter: pickup in Nov-Feb (higher disruption risk)
# - volume_to_weight: proxy for irregular/awkward parcels
# - carrier_route_risk: carrier reliability * route historical delay rate
# - day_of_week: Monday/Friday shipments may face different delays
# - is_cross_region: same vs different region (local vs long-haul)

# Your code here

***(6 marks)* Explain your new features.**

For each feature, explain what it measures and why it should predict delays.

*Your answer here.*

---
# SECTION 6 - Training Models *(15 marks)*

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

EXCLUDE = ['shipment_id','pickup_date','origin_region','dest_region',
           'carrier_id','route_id','is_delayed']
# Add any categorical columns you haven't encoded yet

feature_cols = [c for c in master.columns if c not in EXCLUDE]
X = master[feature_cols]
y = master['is_delayed']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f'Train: {len(X_train):,}  Val: {len(X_val):,}  Delay rate val: {y_val.mean():.2%}')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = {}

In [ ]:
# *(3 marks)* Model 1 - Logistic Regression
from sklearn.linear_model import LogisticRegression

# Your code here

***(2 marks)* Model 1 - explain your choices.**

- Why LR for delay prediction?
- How did you handle class imbalance?
- What regularisation and why?

*Your answer here.*

In [ ]:
# *(3 marks)* Model 2 - Random Forest or LightGBM

# Your code here

***(2 marks)* Model 2 - explain your choices.**

*Your answer here.*

In [ ]:
# *(2 marks)* Compare models. Print CV AUC, Val AUC, overfit gap.

# Your code here

best_prob  = None
best_model = None

***(3 marks)* Which model did you choose and why?**

*Your answer here.*

---
# SECTION 7 - Evaluating Your Model *(8 marks)*

In [ ]:
# *(8 marks)* Produce:
# 1. ROC curves for all models
# 2. Score distribution by delay=0 vs delay=1
# 3. Feature importance / coefficients (top 15)

# Your code here

**Interpret your diagnostics.**

*Your answer here.*

---
# SECTION 8 - Business Impact *(12 marks)*

In [ ]:
# *(7 marks)* Compute the delay capture rate at 15% intervention.
#
# Delay capture rate = (# actual delays in top-15% flagged) / (# total actual delays)
#
# Compare:
#   (a) Random 15% policy
#   (b) Model top-15% policy
#
# Also compute: SLA penalty avoided.
# Assume each avoided delay saves £45 in SLA penalties + handling costs.

# Your code here

***(5 marks)* What is the business impact?**

Explain in plain language the value of flagging high-risk shipments vs random selection.
What would you recommend to SwiftRoute's Operations team?

*Your answer here.*

---
## 8.2 - Save Your Model for Deployment

In [ ]:
import joblib

joblib.dump(best_model, 'model.pkl')
print('Model saved to model.pkl')

---
# SECTION 9 - Monitoring for Drift in Production *(5 marks)*

**9.1** *(2 marks)* Propose how you would detect **input drift** on the live shipment stream. Which features are most likely to drift and what tests would you use?

*Your answer here.*

---

**9.2** *(3 marks)* Propose how you would detect **target drift** - i.e. the actual delay rate changing in ways the model no longer predicts correctly. What would trigger retraining?

*Your answer here.*

---
# EXTENSION *(10 marks)*

Attempt **both** tasks (5 marks each).

**E1** *(5 marks)* **Route-level analysis**: cluster routes by risk profile. Can you identify systematic weaknesses in SwiftRoute's network?

**E2** *(5 marks)* **SHAP**: for the top 10 highest-risk shipments in the validation set, use SHAP to explain exactly which factors are driving each prediction.